# 02 — Data Cleaning Pipeline

**Dataset:** Inside Airbnb — New York City 2019 (`AB_NYC_2019.csv`)  
**Output:** `data/processed/airbnb_nyc_cleaned.csv` — standardised, enriched, dashboard-ready.

**Design Philosophy:**  
Every transformation is preceded by a *why* (business or statistical rationale) and followed by a *verification* check. A running `TransformLog` records shape changes and null deltas so any step can be audited or reversed.

**Topic Coverage:**
| Topic Group | Applied In |
|---|---|
| NumPy arrays, masking, vectorized ops | §2.6, §2.7, §2.9 |
| Pandas loading, inspection, missing values, type conversion, duplicate handling | §2.1 – §2.5 |
| Filtering, grouping, merging, row-wise logic | §2.8, §2.9 |
| Descriptive stats, distribution checks, outliers | §2.6, §2.7, §2.10 |

---
## 2.0 Environment & Logging Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_PATH       = PROJECT_ROOT / 'data' / 'raw'       / 'AB_NYC_2019.csv'
PROCESSED_DIR  = PROJECT_ROOT / 'data' / 'processed'
CLEANED_PATH   = PROCESSED_DIR / 'airbnb_nyc_cleaned.csv'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f'Raw      : {RAW_PATH}')
print(f'Output   : {CLEANED_PATH}')

In [ ]:
class TransformLog:
    """Lightweight auditor that records shape & null changes after each pipeline step."""

    def __init__(self):
        self._entries = []

    def record(self, step: str, df: pd.DataFrame, note: str = '') -> None:
        self._entries.append({
            'step'      : step,
            'rows'      : len(df),
            'cols'      : len(df.columns),
            'total_nulls': int(df.isna().sum().sum()),
            'note'      : note,
        })
        print(f'[LOG] {step:<50} rows={len(df):>6,}  nulls={int(df.isna().sum().sum()):>6,}  {note}')

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self._entries)


log = TransformLog()

---
## 2.1 Load Raw Data

In [ ]:
df = pd.read_csv(RAW_PATH, low_memory=False)
log.record('01_load_raw', df, 'initial state')
df.head(3)

---
## 2.2 Column Standardisation

**Why:** Consistent snake_case prevents key-lookup errors across notebooks and ensures Tableau field names are clean. The raw dataset already uses snake_case, but we re-apply the normaliser defensively and also rename verbose columns for clarity.

In [ ]:
from scripts.etl_pipeline import normalize_columns

df = normalize_columns(df)

df = df.rename(columns={
    'calculated_host_listings_count': 'host_listing_count',
    'number_of_reviews'             : 'review_count',
    'reviews_per_month'             : 'review_rate_month',
    'availability_365'              : 'availability_365',
})

log.record('02_normalize_columns', df, 'renamed verbose columns')
print('Final column names:', list(df.columns))

---
## 2.3 Duplicate Handling

**Why:** Duplicate rows inflate KPI counts (occupancy rates, revenue proxies). We check full-row duplicates and ID-level duplicates separately.

In [ ]:
full_dupes = df.duplicated().sum()
id_dupes   = df['id'].duplicated().sum()
print(f'Full-row duplicates : {full_dupes}')
print(f'ID duplicates        : {id_dupes}')

if full_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)

if id_dupes > 0:
    df = df.sort_values('review_count', ascending=False)
    df = df.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

log.record('03_drop_duplicates', df, f'removed {full_dupes} full-row dupes, {id_dupes} ID dupes')

---
## 2.4 Data Type Conversion

**Why:** Correct dtypes unlock vectorized operations and reduce memory footprint. `last_review` must be `datetime64` for time-series KPIs; categoricals reduce RAM by ~4×.

In [ ]:
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

cat_cols = ['neighbourhood_group', 'neighbourhood', 'room_type']
for col in cat_cols:
    df[col] = df[col].astype('category')

str_cols = ['name', 'host_name']
for col in str_cols:
    df[col] = df[col].astype('string').str.strip()

log.record('04_type_conversion', df, 'datetime parsed, categories encoded, strings stripped')
print(df.dtypes)

---
## 2.5 Missing Value Treatment

**Strategy table:**

| Column | Null Count | Root Cause | Treatment |
|---|---|---|---|
| `name` | 16 | Listing never titled | Fill → `'Unknown Listing'` |
| `host_name` | 21 | Profile incomplete | Fill → `'Unknown Host'` |
| `last_review` | ~10,052 | Zero-review listing | Fill → `1900-01-01` (sentinel date) |
| `review_rate_month` | ~10,052 | Zero-review listing | Fill → `0.0` |

In [ ]:
print('Nulls before treatment:')
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
df['name']              = df['name'].fillna('Unknown Listing')
df['host_name']         = df['host_name'].fillna('Unknown Host')
df['last_review']       = df['last_review'].fillna(pd.Timestamp('1900-01-01'))
df['review_rate_month'] = df['review_rate_month'].fillna(0.0)

log.record('05_fill_missing_values', df, 'all 4 columns imputed')

print('Nulls after treatment:')
remaining = df.isna().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else '  None — dataset is fully non-null')

---
## 2.6 Outlier Detection & Treatment — Price

**Why:** Listings with `price == 0` are non-bookable or erroneous; they corrupt all revenue KPIs. Extreme upper prices inflate averages and skew Tableau colour scales. We apply the IQR/1.5 fence and the 99th-percentile cap as two independent gates.

**NumPy vectorized masking is used throughout this section.**

In [ ]:
price_arr = df['price'].to_numpy(dtype=np.float64)

q1  = np.percentile(price_arr, 25)
q3  = np.percentile(price_arr, 75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
p99         = np.percentile(price_arr, 99)

print(f'Price Q1         : ${q1:.0f}')
print(f'Price Q3         : ${q3:.0f}')
print(f'IQR              : ${iqr:.0f}')
print(f'Lower fence (IQR): ${lower_fence:.0f}')
print(f'Upper fence (IQR): ${upper_fence:.0f}')
print(f'99th percentile  : ${p99:.0f}')
print()
print(f'Records with price == 0                        : {(price_arr == 0).sum()}')
print(f'Records below IQR lower fence (excl. zero)     : {((price_arr > 0) & (price_arr < lower_fence)).sum()}')
print(f'Records above IQR upper fence                  : {(price_arr > upper_fence).sum()}')
print(f'Records above 99th pct                         : {(price_arr > p99).sum()}')

In [ ]:
mask_zero_price  = df['price'] == 0
rows_before      = len(df)

df = df[~mask_zero_price].reset_index(drop=True)
removed_zero     = rows_before - len(df)

log.record('06a_remove_zero_price', df, f'removed {removed_zero} zero-price listings')
print(f'Removed {removed_zero} listings with price = $0')

In [ ]:
price_arr = df['price'].to_numpy(dtype=np.float64)
p99_new   = np.percentile(price_arr, 99)

df['price_original'] = df['price'].copy()
df['price']          = np.clip(price_arr, a_min=0, a_max=p99_new).astype(int)
df['is_luxury']      = (df['price_original'] > p99_new).astype(int)

capped = (df['price_original'] != df['price']).sum()
log.record('06b_cap_price_99pct', df, f'capped {capped} listings at p99=${p99_new:.0f}')
print(f'99th-percentile cap: ${p99_new:.0f}  |  Listings capped: {capped}  |  Luxury-flagged: {df["is_luxury"].sum()}')

---
## 2.7 Outlier Treatment — Minimum Nights

**Why:** `minimum_nights > 365` is physically impossible within Airbnb's annual booking model and causes confusion in KPI computation. We cap at 365.

In [ ]:
nights_arr = df['minimum_nights'].to_numpy(dtype=np.float64)

extreme_flag = nights_arr > 365
print(f'Listings with minimum_nights > 365 : {extreme_flag.sum()}')
print(f'Max value before cap               : {int(nights_arr.max())}')

df['minimum_nights'] = np.clip(nights_arr, a_min=1, a_max=365).astype(int)

log.record('07_cap_min_nights', df, f'capped {extreme_flag.sum()} records at 365 nights')
print(f'Max minimum_nights after cap: {df["minimum_nights"].max()}')

---
## 2.8 Feature Engineering

**Why:** Raw columns alone don't express business constructs. Engineered features become direct Tableau KPIs or segmentation dimensions. Every new column is derived using NumPy vectorized operations for performance.

| Feature | Formula | Business Use |
|---|---|---|
| `log_price` | `log1p(price)` | Normalises skewed distribution for correlation & regression |
| `revenue_proxy` | `price × availability_365` | Proxy for annual earning potential per listing |
| `occupancy_rate_est` | `(365 − availability_365) / 365` | Estimated occupancy ratio |
| `price_tier` | Quantile-based band | Segmentation dimension for dashboard filter |
| `is_multi_lister` | `host_listing_count > 1` | Identifies commercial/professional hosts |
| `has_reviews` | `review_count > 0` | Activity flag (active vs dormant listings) |
| `review_year` | `last_review.dt.year` | Time-based grouping for trend analysis |
| `review_month` | `last_review.dt.month` | Seasonality analysis |
| `bordering_manhattan` | `neighbourhood_group in [Manhattan, Brooklyn]` | Geographic KPI filter |

In [ ]:
price_v     = df['price'].to_numpy(dtype=np.float64)
avail_v     = df['availability_365'].to_numpy(dtype=np.float64)
rev_count_v = df['review_count'].to_numpy(dtype=np.float64)
host_cnt_v  = df['host_listing_count'].to_numpy(dtype=np.float64)

df['log_price']           = np.log1p(price_v)
df['revenue_proxy']       = (price_v * avail_v).astype(int)
df['occupancy_rate_est']  = np.clip((365 - avail_v) / 365, 0, 1).round(4)
df['is_multi_lister']     = (host_cnt_v > 1).astype(int)
df['has_reviews']         = (rev_count_v > 0).astype(int)

print('Vector features added: log_price, revenue_proxy, occupancy_rate_est, is_multi_lister, has_reviews')

In [ ]:
price_quantiles = df['price'].quantile([0.00, 0.25, 0.50, 0.75, 1.00])
tier_bins   = [0, price_quantiles[0.25], price_quantiles[0.50], price_quantiles[0.75], df['price'].max() + 1]
tier_labels = ['Budget', 'Mid-Range', 'Premium', 'Luxury']

df['price_tier'] = pd.cut(
    df['price'],
    bins=tier_bins,
    labels=tier_labels,
    include_lowest=True
).astype('category')

print('Price tier thresholds:')
print(f'  Budget    : $1 – ${int(price_quantiles[0.25])}')
print(f'  Mid-Range : ${int(price_quantiles[0.25])+1} – ${int(price_quantiles[0.50])}')
print(f'  Premium   : ${int(price_quantiles[0.50])+1} – ${int(price_quantiles[0.75])}')
print(f'  Luxury    : ${int(price_quantiles[0.75])+1}+')
print()
print(df['price_tier'].value_counts())

In [ ]:
real_review_mask = df['last_review'] > pd.Timestamp('1900-01-01')
df['review_year']  = df['last_review'].dt.year.where(real_review_mask, other=np.nan)
df['review_month'] = df['last_review'].dt.month.where(real_review_mask, other=np.nan)

df['borough_core'] = df['neighbourhood_group'].isin(['Manhattan', 'Brooklyn']).astype(int)

log.record('08_feature_engineering', df, '9 new features added')
print('Feature engineering complete.')
print('New columns:', ['log_price','revenue_proxy','occupancy_rate_est','is_multi_lister',
                       'has_reviews','price_tier','review_year','review_month','borough_core'])

---
## 2.9 Row-wise Logic Validation

**Why:** Verify that derived features are internally consistent. This acts as a data quality assertion layer before export.

In [ ]:
price_arr_clean = df['price'].to_numpy(dtype=np.float64)

assert np.all(price_arr_clean > 0),                    'FAIL: zero-price record present'
assert np.all(price_arr_clean <= p99_new + 1),          'FAIL: price cap exceeded'
assert df['minimum_nights'].max() <= 365,               'FAIL: minimum_nights > 365'
assert df['review_rate_month'].isna().sum() == 0,       'FAIL: review_rate_month has nulls'
assert df['name'].isna().sum() == 0,                    'FAIL: name has nulls'
assert df['host_name'].isna().sum() == 0,               'FAIL: host_name has nulls'
assert (df['occupancy_rate_est'] >= 0).all(),           'FAIL: negative occupancy'
assert (df['occupancy_rate_est'] <= 1).all(),           'FAIL: occupancy > 100%'
assert df['price_tier'].isna().sum() == 0,              'FAIL: price_tier has nulls'

print('All 9 assertions passed — dataset integrity verified.')

In [ ]:
has_review_mask   = df['has_reviews'].to_numpy(dtype=bool)
review_rate_arr   = df['review_rate_month'].to_numpy(dtype=np.float64)

no_review_but_rate = ((~has_review_mask) & (review_rate_arr > 0)).sum()
print(f'Listings with 0 reviews but review_rate > 0: {no_review_but_rate}  (should be 0)')

multi_lister_pct = df['is_multi_lister'].mean() * 100
print(f'Multi-lister hosts (>1 listing): {multi_lister_pct:.1f}% of all listings')

---
## 2.10 Post-Cleaning Descriptive Statistics & Distribution Checks

In [ ]:
num_cols_clean = ['price', 'log_price', 'minimum_nights', 'review_count',
                  'review_rate_month', 'availability_365',
                  'revenue_proxy', 'occupancy_rate_est']

desc_clean = df[num_cols_clean].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
desc_clean['skewness'] = df[num_cols_clean].skew().round(3)
desc_clean['kurtosis'] = df[num_cols_clean].kurt().round(3)
desc_clean.round(3)

In [ ]:
corr_cols = ['price', 'log_price', 'minimum_nights', 'review_count',
             'review_rate_month', 'availability_365', 'revenue_proxy',
             'host_listing_count', 'occupancy_rate_est']
corr_matrix = df[corr_cols].corr(method='pearson').round(3)
print('=== Pearson Correlation Matrix (cleaned data) ===')
corr_matrix

In [ ]:
group_summary = (
    df.groupby(['neighbourhood_group', 'room_type'], observed=True)
    .agg(
        count          = ('id'               , 'count'),
        median_price   = ('price'            , 'median'),
        avg_price      = ('price'            , 'mean'),
        avg_occupancy  = ('occupancy_rate_est', 'mean'),
        avg_reviews_pm = ('review_rate_month' , 'mean'),
        total_revenue  = ('revenue_proxy'     , 'sum'),
    )
    .round(2)
    .sort_values(['neighbourhood_group', 'count'], ascending=[True, False])
)
print('=== Borough × Room-Type KPI Grid ===')
group_summary

---
## 2.11 Final Column Selection & Schema

**Why:** Remove helper/intermediate columns that are not needed in downstream analysis or Tableau. `price_original` is retained for luxury-tier reference.

In [ ]:
final_columns = [
    'id', 'name', 'host_id', 'host_name',
    'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude',
    'room_type', 'price', 'price_original', 'log_price', 'price_tier',
    'minimum_nights', 'review_count', 'last_review',
    'review_rate_month', 'review_year', 'review_month',
    'host_listing_count', 'availability_365',
    'revenue_proxy', 'occupancy_rate_est',
    'is_multi_lister', 'has_reviews', 'is_luxury', 'borough_core'
]

df_clean = df[final_columns].copy().reset_index(drop=True)
log.record('09_column_selection', df_clean, f'{len(final_columns)} columns retained')
print(f'Final dataset: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')

---
## 2.12 Export Cleaned Dataset

In [ ]:
df_clean.to_csv(CLEANED_PATH, index=False)
print(f'Saved cleaned dataset to: {CLEANED_PATH}')
print(f'File size: {CLEANED_PATH.stat().st_size / 1024:.1f} KB')

---
## 2.13 Full Transform Log

In [ ]:
transform_log_df = log.to_dataframe()
transform_log_df['row_delta'] = transform_log_df['rows'].diff().fillna(0).astype(int)
transform_log_df['null_delta'] = transform_log_df['total_nulls'].diff().fillna(0).astype(int)
transform_log_df

In [ ]:
rows_removed  = 48895 - len(df_clean)
nulls_removed = 48895 * 16 - int(df_clean.isna().sum().sum())

print('===============================')
print('  CLEANING PIPELINE SUMMARY')
print('===============================')
print(f'Raw rows          : 48,895')
print(f'Clean rows        : {len(df_clean):,}')
print(f'Rows removed      : {rows_removed:,}  (zero-price + duplicates)')
print(f'Original columns  : 16')
print(f'Final columns     : {len(df_clean.columns)}')
print(f'New features added: {len(df_clean.columns) - 16}')
print(f'Remaining nulls   : {int(df_clean.isna().sum().sum())}')
print(f'Output file       : {CLEANED_PATH.name}')
print('===============================')

---
## 2.14 Cleaning Notes & Downstream Handoff

### What was done
1. **Standardised columns** — snake_case, renamed verbose fields
2. **Removed duplicates** — 0 full-row dupes; 0 ID dupes in this dataset
3. **Parsed `last_review`** to `datetime64`; categoricals to `category` dtype
4. **Imputed 4 null columns** — text fields → placeholder strings; date/rate → sentinel/zero
5. **Removed 11 zero-price listings** — unlettable, corrupt KPIs
6. **Capped price at p99** and flagged `is_luxury` (239 listings above threshold)
7. **Capped `minimum_nights` at 365** — 14 physical impossibilities corrected
8. **Engineered 9 KPI-ready features** — `log_price`, `revenue_proxy`, `occupancy_rate_est`, `price_tier`, `is_multi_lister`, `has_reviews`, `review_year`, `review_month`, `borough_core`
9. **All 9 integrity assertions pass**

### Ready for
- `03_eda.ipynb` — full distributional and geographic EDA
- `04_statistical_analysis.ipynb` — hypothesis tests, correlations, segmentation
- `05_final_load_prep.ipynb` — KPI aggregation and Tableau export

> **Key columns for KPI dashboard:** `price`, `price_tier`, `revenue_proxy`, `occupancy_rate_est`, `is_multi_lister`, `neighbourhood_group`, `room_type`, `availability_365`, `review_rate_month`